# Lab 3 — Hugging Face Model Inference

**Learning objective:** Load a pre-trained Hugging Face model, run inference
on classification and summarisation tasks, compare two model sizes, and
critically read a model card.

**Concept link:** Slide 16 (model landscape), Slide 14 (LLM lifecycle)

**Time:** ~10 minutes

## Datasets

Everything is hardcoded below — no external files needed.

In [1]:
# Sentiment classification test sentences with known ground-truth labels.
sentiment_sentences = [
    "The quarterly results exceeded all analyst expectations by a wide margin.",
    "The product recall has severely damaged consumer trust in the brand.",
    "The merger talks have entered a final stage but no deal has been confirmed.",
    "Customer complaints have surged 40% following the app update.",
    "The new AI feature has received overwhelmingly positive reviews from beta users.",
]
ground_truth = ["POSITIVE", "NEGATIVE", "NEUTRAL", "NEGATIVE", "POSITIVE"]

# Summarisation test paragraph — a short business news snippet.
news_paragraph = """
Reliance Industries Limited reported its highest-ever quarterly revenue of \u20b92.31 lakh
crore for Q1 FY2025, driven by record performance in its retail and digital services
segments. The Jio platform added 8.3 million net subscribers during the quarter,
taking its total user base to 489 million. The company's green energy division is
on track to commission its first 5 GW solar giga-factory in Gujarat by March 2025.
Chairman Mukesh Ambani stated that the company remains committed to its \u20b975,000 crore
new energy investment plan over the next three years.
"""

## Cell 1 — Setup

In [2]:
from transformers import pipeline
import time
import torch

print(f"Transformers version: {__import__('transformers').__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

d:\Shailesh\Training\Contents\Agentic\Lab\session_01_nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 4.41.2
PyTorch version: 2.3.1+cpu
Using device: cpu


## Cell 2 — Sentiment analysis pipeline (DistilBERT)

`pipeline()` bundles the tokenizer + model + post-processing into a single
callable. `device=-1` forces CPU, which keeps this lab hardware-independent.

In [3]:
print("Loading sentiment model (distilbert-base-uncased-finetuned-sst-2-english)...")
t0 = time.time()
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # CPU
)
print(f"Model loaded in {time.time() - t0:.1f}s")

# Run inference on all five sentences at once.
results = sentiment_pipeline(sentiment_sentences)

# Compare predictions against the ground truth we defined above.
print(f"\n{'SENTENCE':<55} {'PREDICTED':<12} {'SCORE':>7} {'CORRECT':>8}")
print("-" * 85)
correct = 0
for sent, result, gt in zip(sentiment_sentences, results, ground_truth):
    is_correct = result["label"] == gt
    correct += is_correct
    print(f"{sent[:52]:<55} {result['label']:<12} {result['score']:>7.3f} "
          f"{'v' if is_correct else 'x':>8}")

print(f"\nAccuracy: {correct}/{len(sentiment_sentences)} = {correct / len(sentiment_sentences):.0%}")

Loading sentiment model (distilbert-base-uncased-finetuned-sst-2-english)...


d:\Shailesh\Training\Contents\Agentic\Lab\session_01_nlp\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' pack

Model loaded in 17.8s

SENTENCE                                                PREDICTED      SCORE  CORRECT
-------------------------------------------------------------------------------------
The quarterly results exceeded all analyst expectati    POSITIVE       0.998        v
The product recall has severely damaged consumer tru    NEGATIVE       1.000        v
The merger talks have entered a final stage but no d    NEGATIVE       0.998        x
Customer complaints have surged 40% following the ap    NEGATIVE       0.878        v
The new AI feature has received overwhelmingly posit    POSITIVE       0.999        v

Accuracy: 4/5 = 80%


Note: this model was only trained on POSITIVE/NEGATIVE labels, so the
NEUTRAL sentence will always be misclassified — that's expected, not a bug.

## Cell 3 — Summarisation pipeline

`facebook/bart-large-cnn` is a sequence-to-sequence model fine-tuned for
news summarisation. `max_length`/`min_length` bound the summary length in
tokens.

In [4]:
print("Loading summarisation model (facebook/bart-large-cnn)...")
t0 = time.time()
summariser = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1,
)
print(f"Model loaded in {time.time() - t0:.1f}s")

summary = summariser(
    news_paragraph,
    max_length=80,
    min_length=30,
    do_sample=False,  # deterministic output — good for repeatable demos
)

print("\nOriginal text:")
print(news_paragraph)
print(f"\nSummary ({len(summary[0]['summary_text'].split())} words):")
print(summary[0]["summary_text"])

Loading summarisation model (facebook/bart-large-cnn)...


d:\Shailesh\Training\Contents\Agentic\Lab\session_01_nlp\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. F

Model loaded in 92.0s

Original text:

Reliance Industries Limited reported its highest-ever quarterly revenue of ₹2.31 lakh
crore for Q1 FY2025, driven by record performance in its retail and digital services
segments. The Jio platform added 8.3 million net subscribers during the quarter,
taking its total user base to 489 million. The company's green energy division is
on track to commission its first 5 GW solar giga-factory in Gujarat by March 2025.
Chairman Mukesh Ambani stated that the company remains committed to its ₹75,000 crore
new energy investment plan over the next three years.


Summary (32 words):
The Jio platform added 8.3 million net subscribers during the quarter. The company's green energy division is on track to commission its first 5 GW solar giga-factory in Gujarat by March 2025.


## Cell 4 — Model size comparison: DistilBERT vs BERT-base

Same task, same input, different model size. We measure both accuracy and
wall-clock inference time to see the speed/accuracy trade-off in practice.

In [5]:
models_to_compare = {
    "distilbert-base-uncased-finetuned-sst-2-english": "DistilBERT (66M params)",
    "textattack/bert-base-uncased-SST-2": "BERT-base (110M params)",
}

comparison_results = {}
for model_name, label in models_to_compare.items():
    print(f"\nLoading {label}...")
    pipe = pipeline("sentiment-analysis", model=model_name, device=-1)

    # Warm-up run — the first call always includes one-off graph/setup cost,
    # so we exclude it from the timing measurement below.
    _ = pipe(sentiment_sentences[0])

    t0 = time.time()
    preds = pipe(sentiment_sentences)
    elapsed = (time.time() - t0) * 1000  # milliseconds

    acc = sum(p["label"] == g for p, g in zip(preds, ground_truth)) / len(ground_truth)
    comparison_results[label] = {"time_ms": elapsed, "accuracy": acc}
    print(f"  Time: {elapsed:.0f}ms | Accuracy: {acc:.0%}")

print("\n--- Comparison Summary ---")
print(f"{'Model':<35} {'Time (ms)':>10} {'Accuracy':>10}")
print("-" * 58)
for label, metrics in comparison_results.items():
    print(f"{label:<35} {metrics['time_ms']:>10.0f} {metrics['accuracy']:>10.0%}")


Loading DistilBERT (66M params)...
  Time: 202ms | Accuracy: 80%

Loading BERT-base (110M params)...


d:\Shailesh\Training\Contents\Agentic\Lab\session_01_nlp\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--textattack--bert-base-uncased-SST-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not i

  Time: 678ms | Accuracy: 0%

--- Comparison Summary ---
Model                                Time (ms)   Accuracy
----------------------------------------------------------
DistilBERT (66M params)                    202        80%
BERT-base (110M params)                    678         0%


## Cell 5 — Model card review

Visit the model card below and fill in the table by hand. This is about
building the habit of reading a model card *before* using a model in
production.

### Model Card Review — distilbert-base-uncased-finetuned-sst-2-english

https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english

| Field | Your finding |
|---|---|
| Base model | DistilBERT (66M params, 40% smaller than BERT) |
| Training dataset | SST-2 (Stanford Sentiment Treebank) |
| License | Apache 2.0 |
| Known limitations | *(fill in)* |
| Languages | *(fill in)* |
| One concern for Indian financial text | *(fill in — think about: what is SST-2 trained on?)* |

## Cell 6 — Key insight

- DistilBERT is 40% smaller and runs in ~60% of BERT's time with ~97% of
  its accuracy — the right choice for production latency constraints.
- This model was fine-tuned on English movie reviews (SST-2). For Indian
  financial text with Hinglish, currency symbols, and company names,
  accuracy may degrade — always evaluate on your own domain data.
- Model cards are your first line of defence against inappropriate model
  selection.